# Lab: Positions and headings

You will turn the car's logged positions into arrows, write the heading arrow, find an arrow's direction without
getting it backwards half the time, wrap an angle, and split an arrow into the parts ahead of the car and to its left.
Read the lecture first; this lab points back to its sections by title and follows its order.

Run the cells from top to bottom (Shift+Enter). Each problem has a "Guess first" line: write your guess down before you
run the cell, then compare. `practice` prints a green tick or tells you what it expected.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/result_first.png" alt="The car on the first bend of gentle at step 700, seen from above. Two dots mark the car&#x27;s front axle at steps 700 and 701, 0.40 m apart. An arrow from the front axle shows the way the car points, up and to the left in this picture. A cone stands on the grass to the right of the road. Two braces split the distance to it, one labelled &quot;? m ahead&quot; and one labelled &quot;? m to the side&quot;.">
<figcaption><em>Lecture figure 1. The car on the first bend of gentle at step 700, seen from above. Two dots mark the car&#x27;s front axle at steps 700 and 701, 0.40 m apart. An arrow from the front axle shows the way the car points, up and to the left in this picture. A cone stands on the grass to the right of the road. Two braces split the distance to it, one labelled &quot;? m ahead&quot; and one labelled &quot;? m to the side&quot;.</em></figcaption>
</figure>

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -q "git+https://github.com/vvknyn/self-driving-ai-course@main"
%matplotlib inline

In [ ]:
import math

import numpy as np

from zero2fsd.car import Car, run
from zero2fsd.grade import check, practice
from zero2fsd.sim import BicycleState, make_scenario
from zero2fsd.sim.camera import to_vehicle
from zero2fsd.sim.scenarios import TARGET_SPEED
from zero2fsd.sim.world import wrap_angle as sim_wrap

## 1. Points and arrows

The next cell drives the provided car once round `gentle` and keeps what the lecture used. `P` holds the front axle's
position at every step, shape `(1778, 2)`: column 0 is $x$ (east), column 1 is $y$ (north). `D` holds the arrow from
each position to the next, shape `(1777, 2)`. `yaw` is the car's yaw at every step, in radians. `road` is the road
itself, which you use for the cone in section 5.

In [ ]:
tel = run(Car(), "gentle")
road = make_scenario("gentle").road
P = np.stack([tel.x, tel.y], axis=-1)
D = np.diff(P, axis=0)
yaw = tel.yaw
print("positions:", P.shape, " step arrows:", D.shape)

**Problem 1.** Find the length of every step arrow with Pythagoras (`np.hypot` does it), as `L`, a vector of 1777
lengths. Then read off the length of the step-700 arrow and of the shortest step on the lap.

**Guess first:** the car drives 8 m/s and the simulator steps 0.05 s. How long is each arrow, and will the shortest
differ from the others?

In [ ]:
L = None  # your code: the length of every step arrow, from D[:, 0] and D[:, 1]

if L is not None:
    practice("length of the step-700 arrow (m)", L[700], 0.4000, tol=1e-4)
    practice("shortest step on the lap (m)", L.min(), 0.39999947, tol=1e-8)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/world_lap.png" alt="The gentle lap from above in the world frame, with the x axis pointing right and the y axis pointing up from the origin where the road starts. The car starts at (0, −1.8), facing along +x, and drives anticlockwise. Over the lap its x runs from −81.3 m to 181.3 m and its y from −1.8 m to 161.8 m. A dot marks the car at step 700, on the first bend.">
<figcaption><em>Lecture figure 2. The gentle lap from above in the world frame, with the x axis pointing right and the y axis pointing up from the origin where the road starts. The car starts at (0, −1.8), facing along +x, and drives anticlockwise. Over the lap its x runs from −81.3 m to 181.3 m and its y from −1.8 m to 161.8 m. A dot marks the car at step 700, on the first bend.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/step_arrow.png" alt="Zoomed in on the front axle at steps 700 and 701. Two dots mark the front axle at steps 700 and 701, and an arrow joins them. Its x part, −0.241 m, and its y part, +0.319 m, are drawn as the two short sides of a right-angled triangle, and the arrow is 0.400 m long. The same arrow is drawn twice more, unchanged, at two other places on the picture.">
<figcaption><em>Lecture figure 3. Zoomed in on the front axle at steps 700 and 701. Two dots mark the front axle at steps 700 and 701, and an arrow joins them. Its x part, −0.241 m, and its y part, +0.319 m, are drawn as the two short sides of a right-angled triangle, and the arrow is 0.400 m long. The same arrow is drawn twice more, unchanged, at two other places on the picture.</em></figcaption>
</figure>

## 2. The heading arrow

**Problem 2.** Write `heading_vector(theta)`: the 1 m arrow $(\cos\theta, \sin\theta)$ at angle `theta`. It must take
one angle or an array of angles, and put the two parts in the last axis, so an array of 3 angles gives shape `(3, 2)`.

**Guess first:** the yaw at step 700 is 2.1812 rad, a little past a quarter turn. Is the east part positive or
negative?

In [ ]:
def heading_vector(theta):
    """The 1 m arrow (cos theta, sin theta); shape (..., 2) for angles of shape (...)."""
    return None  # your code: np.asarray(theta, float), then np.stack([...], axis=-1)

practice("heading arrow at step 700", heading_vector(yaw[700]), np.array([-0.5732, 0.8194]), tol=1e-4)
practice("shape for three angles", None if heading_vector(np.zeros(3)) is None else heading_vector(np.zeros(3)).shape, (3, 2))

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/theta_psi.png" alt="The car at step 700 on the first bend, seen from above. A thin line from the front axle points east, and a curved arrow from it to the way the car points is labelled θ = 2.1812 rad, more than a quarter turn. A dashed line shows the road&#x27;s direction there, and the small angle from it to the way the car points is labelled ψ = −0.0325 rad, drawn larger than it is.">
<figcaption><em>Lecture figure 4. The car at step 700 on the first bend, seen from above. A thin line from the front axle points east, and a curved arrow from it to the way the car points is labelled θ = 2.1812 rad, more than a quarter turn. A dashed line shows the road&#x27;s direction there, and the small angle from it to the way the car points is labelled ψ = −0.0325 rad, drawn larger than it is.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/unit_circle.png" alt="A circle of radius 1 m around the front axle, with east to the right and north up. Three thin arrows point at θ = 0, π/2 and π and end at (1, 0), (0, 1) and (−1, 0). A thick arrow points at θ = 2.1812, the car&#x27;s yaw at step 700. Its east part, −0.5732 m, and its north part, +0.8194 m, are drawn as the two short sides of a right-angled triangle.">
<figcaption><em>Lecture figure 5. A circle of radius 1 m around the front axle, with east to the right and north up. Three thin arrows point at θ = 0, π/2 and π and end at (1, 0), (0, 1) and (−1, 0). A thick arrow points at θ = 2.1812, the car&#x27;s yaw at step 700. Its east part, −0.5732 m, and its north part, +0.8194 m, are drawn as the two short sides of a right-angled triangle.</em></figcaption>
</figure>

## 3. Which way is it pointing

**Problem 3.** Compute the angle the lecture first tried, `np.arctan(dy / dx)`, for every step arrow, as `naive`.
Then count the steps where it misses the car's yaw by more than 1 rad. Use `sim_wrap`, the simulator's own wrap, on the
difference: you write your own in section 4. (Dividing by $\Delta x = 0$ is harmless here, so the cell silences the
warning.)

**Guess first:** what share of the 1777 steps will be more than 1 rad off: none, a few, or about half?

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    naive = None  # your code: np.arctan of D[:, 1] / D[:, 0]

if naive is not None:
    n_off = int((np.abs(sim_wrap(naive - yaw[:-1])) > 1).sum())
    practice("steps more than 1 rad off", n_off, 888, tol=0)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/naive_700.png" alt="The step-700 arrow from the zoom above, pointing up and to the left, north-west. Beside it, a 1 m arrow at the angle arctan of the slope gives, −0.9241 rad, pointing down and to the right, south-east. The two arrows point in opposite directions.">
<figcaption><em>Lecture figure 6. The step-700 arrow from the zoom above, pointing up and to the left, north-west. Beside it, a 1 m arrow at the angle arctan of the slope gives, −0.9241 rad, pointing down and to the right, south-east. The two arrows point in opposite directions.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/naive_lap.png" alt="The angle between the arrow that arctan of the slope gives and the car&#x27;s heading arrow, at every step of the gentle lap, plotted against the time into the lap. It is small on every step that moves with an east part, and close to π, pointing backwards, on every step that moves west. A dashed line marks 1 rad.">
<figcaption><em>Lecture figure 7. The angle between the arrow that arctan of the slope gives and the car&#x27;s heading arrow, at every step of the gentle lap, plotted against the time into the lap. It is small on every step that moves with an east part, and close to π, pointing backwards, on every step that moves west. A dashed line marks 1 rad.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/opposite_arrows.png" alt="Two arrows from one point: (1, 1), pointing north-east, and (−1, −1), pointing south-west. Their slopes are equal, 1, so arctan gives both the same angle, 0.7854 rad, which is north-east.">
<figcaption><em>Lecture figure 8. Two arrows from one point: (1, 1), pointing north-east, and (−1, −1), pointing south-west. Their slopes are equal, 1, so arctan gives both the same angle, 0.7854 rad, which is north-east.</em></figcaption>
</figure>

**Exercise A.** Write `direction_angle(v)`, the direction of an arrow `v` with `np.arctan2`. Two details matter: `v` can
be one arrow of shape `(2,)` or many of shape `(N, 2)`, and the zero arrow has no direction, so it must give `nan`, not
0.

In [ ]:
def direction_angle(v):
    """The direction of an arrow, in radians anticlockwise from east, in [-pi, pi].

    Input:   v, an arrow (v_x, v_y) of shape (2,), or many arrows of shape (N, 2).
    Output:  the angle(s), a float for one arrow and shape (N,) for many; nan for the zero arrow (0, 0).
    Example: direction_angle(np.array([3.0, 4.0]))  ->  0.9272952180016122
    """
    raise NotImplementedError

**Guess first (Exercise A):** how far from the yaw is your direction in the worst step of the lap: about 0.04 rad, 0.4
rad, or 4 rad?

In [ ]:
check("1.1.1.a", direction_angle)
try:
    practice("direction of (3, 4)", direction_angle(np.array([3.0, 4.0])), 0.92729522, tol=1e-8)
    practice("the zero arrow gives nan", bool(np.isnan(direction_angle(np.zeros(2)))), True)
    practice("worst miss over the lap (rad)", np.abs(sim_wrap(direction_angle(D) - yaw[:-1])).max(), 0.040994, tol=1e-6)
except NotImplementedError:
    print("Write direction_angle above first, then run this cell again.")

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/atan2_lap.png" alt="The same comparison as before, for atan2, in two panels, one above the other: the angle between the arrow at atan2&#x27;s answer and the car&#x27;s heading arrow, at every step of the gentle lap, against the time into the lap. Top: at the scale of the arctan curve from before, which is drawn faded behind it, atan2&#x27;s curve lies along 0. Bottom: the atan2 curve alone, magnified. It is never more than 0.041 rad, and it stays between 0.029 and 0.041 rad through the middles of both bends.">
<figcaption><em>Lecture figure 9. The same comparison as before, for atan2, in two panels, one above the other: the angle between the arrow at atan2&#x27;s answer and the car&#x27;s heading arrow, at every step of the gentle lap, against the time into the lap. Top: at the scale of the arctan curve from before, which is drawn faded behind it, atan2&#x27;s curve lies along 0. Bottom: the atan2 curve alone, magnified. It is never more than 0.041 rad, and it stays between 0.029 and 0.041 rad through the middles of both bends.</em></figcaption>
</figure>

## 4. Angles go round

**Problem 4.** Subtract the yaw from your direction, `direction_angle(D) - yaw[:-1]`, with no wrap, and count the
steps where the result is more than 1 rad from 0.

**Guess first:** the directions are within 0.04 rad of the yaw (Exercise A). So how many steps differ by more than 1
rad: 0, or a lot?

In [ ]:
n_plain = None  # your code: the number of steps where |direction_angle(D) - yaw[:-1]| > 1

practice("steps more than 1 rad off, unwrapped", n_plain, 722, tol=0)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/raw_diff.png" alt="The plain difference, atan2&#x27;s answer minus the yaw, at every step of the gentle lap, with thin lines joining neighbouring steps. It stays near 0 until the back straight, where the car points west. There it jumps back and forth between near 0 and near −2π, −6.28, and after it stays near −2π: 722 of the 1777 steps are more than 1 rad from 0. The inset zooms on two seconds of the back straight, where each step lands on one line or the other.">
<figcaption><em>Lecture figure 10. The plain difference, atan2&#x27;s answer minus the yaw, at every step of the gentle lap, with thin lines joining neighbouring steps. It stays near 0 until the back straight, where the car points west. There it jumps back and forth between near 0 and near −2π, −6.28, and after it stays near −2π: 722 of the 1777 steps are more than 1 rad from 0. The inset zooms on two seconds of the back straight, where each step lands on one line or the other.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/yaw_wrapped.png" alt="Two panels, one above the other, against the distance along the road. Top: the yaw as the simulator stores it, climbing from 0 to 6.2264 rad over the gentle lap. Bottom: the same yaw brought into the range from −π to π by taking away whole turns. On the back straight, from 403.9 m to 442.7 m, the yaw wobbles across π 39 times, and each time the bottom curve jumps between π and −π. It ends at −0.0568 rad.">
<figcaption><em>Lecture figure 11. Two panels, one above the other, against the distance along the road. Top: the yaw as the simulator stores it, climbing from 0 to 6.2264 rad over the gentle lap. Bottom: the same yaw brought into the range from −π to π by taking away whole turns. On the back straight, from 403.9 m to 442.7 m, the yaw wobbles across π 39 times, and each time the bottom curve jumps between π and −π. It ends at −0.0568 rad.</em></figcaption>
</figure>

**Problem 5.** Write `wrap_angle(a)`: the same angle shifted by whole turns into $[-\pi, \pi)$. Use
$(a + \pi) \bmod 2\pi - \pi$ with Python's `%` (not `math.fmod`, which keeps the sign of its input).

**Guess first:** the yaw at the end of the lap is 6.226408 rad, just under a full turn. Will its wrap be near 0, near
$\pi$, or near $2\pi$?

In [ ]:
def wrap_angle(a):
    """The angle a shifted by whole turns into [-pi, pi)."""
    return None  # your code: (a + math.pi) % (2 * math.pi) - math.pi

probe = np.concatenate([np.random.default_rng(1111).uniform(-40, 40, 100_000), np.arange(-8, 9) * math.pi, [math.pi, -math.pi, 0.0, 1e-300]])

practice("wrap of the final yaw", wrap_angle(yaw[-1]), -0.056778, tol=1e-6)
practice("wrap of pi", wrap_angle(np.pi), -np.pi, tol=1e-12)
practice("same as the simulator's wrap", None if wrap_angle(probe) is None else bool(np.array_equal(wrap_angle(probe), sim_wrap(probe))), True)

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/wrap_saw.png" alt="The function wrap plotted for angles from −10 to 10 rad: a saw-tooth that climbs with slope 1 and drops by 2π each time it reaches π, so every output lies from −π up to, but not including, π. Dots mark wrap of 6.2264, which is −0.0568, and wrap of π, which is −π.">
<figcaption><em>Lecture figure 12. The function wrap plotted for angles from −10 to 10 rad: a saw-tooth that climbs with slope 1 and drops by 2π each time it reaches π, so every output lies from −π up to, but not including, π. Dots mark wrap of 6.2264, which is −0.0568, and wrap of π, which is −π.</em></figcaption>
</figure>

## 5. Ahead and to the left

The cone is near the road, 12 m further along than the car at step 700 and a few metres off the road's centre line. The
next cell finds it from the road; the lecture rounds it to $(159.131, 140.083)$ but the lab keeps the exact values.

In [ ]:
s_car, _ = road.project(tel.x[700], tel.y[700])
cone = road.offset(s_car + 12.0, -4.3)
print("cone:", cone)

**Guess first:** the car at step 700 faces north-west. For a different car, facing exactly north,
$\theta = \pi/2$, an arrow of 2 m east and 3 m north: how far ahead is it, and how far to the left?

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/north_hand.png" alt="Two panels, one above the other, with the same cone arrow, 2 m east and 3 m north of the front axle. Top: the car faces east, and the cone is 2 m ahead of it and 3 m to its left. Bottom: the car faces north, and the cone is 3 m ahead of it and 2 m to its right. In both, a 1 m arrow shows the way the car points and a dashed line marks the line it drives along.">
<figcaption><em>Lecture figure 13. Two panels, one above the other, with the same cone arrow, 2 m east and 3 m north of the front axle. Top: the car faces east, and the cone is 2 m ahead of it and 3 m to its left. Bottom: the car faces north, and the cone is 3 m ahead of it and 2 m to its right. In both, a 1 m arrow shows the way the car points and a dashed line marks the line it drives along.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/left_arrow.png" alt="The car at step 700 with its heading arrow, (−0.5732, +0.8194), and its left arrow, (−0.8194, −0.5732), both 1 m long and drawn from the front axle, with a right-angle mark between them. A curved arrow shows the quarter turn anticlockwise that takes the first to the second.">
<figcaption><em>Lecture figure 14. The car at step 700 with its heading arrow, (−0.5732, +0.8194), and its left arrow, (−0.8194, −0.5732), both 1 m long and drawn from the front axle, with a right-angle mark between them. A curved arrow shows the quarter turn anticlockwise that takes the first to the second.</em></figcaption>
</figure>

**Exercise B.** Write `forward_left(theta, v)`: the forward part $\mathbf{v} \cdot \hat h$ and the left part
$\hat h \times \mathbf{v} = \cos\theta\, v_y - \sin\theta\, v_x$ of the world arrow `v`, for a car with yaw `theta`. It must
work for one yaw and one arrow, and for arrays: `theta` of shape `(N,)` with `v` of shape `(N, 2)` gives `(N, 2)`.

In [ ]:
def forward_left(theta, v):
    """The forward and left parts of the world arrow v, for a car whose yaw is theta.

    Input:   theta, the yaw in radians (shape (...)); v, the arrow (v_x, v_y) in the world frame (shape (..., 2)).
    Output:  shape (..., 2): column 0 is how far ahead of the car, column 1 how far to its left (negative: to its right).
    Example: forward_left(np.pi / 2, np.array([2.0, 3.0]))  ->  array([ 3., -2.])
    """
    raise NotImplementedError

**Guess first (Exercise B):** the cone is 12 m ahead along the road but the car is turning left. Is the cone's left part
positive or negative?

In [ ]:
check("1.1.1.b", forward_left)
try:
    practice("facing north", forward_left(np.pi / 2, np.array([2.0, 3.0])), np.array([3.0, -2.0]), tol=1e-12)
    practice("the cone at step 700", forward_left(yaw[700], cone - P[700]), np.array([12.6579, -1.6678]), tol=1e-4)
    practice("same as the camera's to_vehicle", forward_left(yaw[700], cone - P[700]),
             np.array(to_vehicle(BicycleState(tel.x[700], tel.y[700], yaw[700], TARGET_SPEED, 0.0), *cone)), tol=1e-9)
except NotImplementedError:
    print("Write forward_left above first, then run this cell again.")

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/cone_parts.png" alt="The car at step 700 with its heading arrow and left arrow drawn from the front axle, and the arrow to the cone, 12.77 m long, whose east part is −5.889 m and north part +11.328 m. A short bar from the cone meets the dashed line along the heading arrow at a right angle, splitting the arrow to the cone into a forward part of 12.66 m along the heading arrow and a left part of −1.67 m, 1.67 m to the right.">
<figcaption><em>Lecture figure 15. The car at step 700 with its heading arrow and left arrow drawn from the front axle, and the arrow to the cone, 12.77 m long, whose east part is −5.889 m and north part +11.328 m. A short bar from the cone meets the dashed line along the heading arrow at a right angle, splitting the arrow to the cone into a forward part of 12.66 m along the heading arrow and a left part of −1.67 m, 1.67 m to the right.</em></figcaption>
</figure>

## 6. Moving is not pointing

**Problem 6.** The lecture's rule says a step's direction is $\theta + \delta + \tfrac{1}{2}\Delta\theta$: the yaw at the
start, plus the wheel angle $\delta$ (`tel.steer[k + 1]` for step $k$), plus half the yaw change over the step. Build
`pred` for every step and find its largest miss against `direction_angle(D)`, wrapping the difference.

**Guess first:** will the rule match to within 0.01 rad, or to within a hair of rounding?

In [ ]:
pred = None  # your code: yaw[:-1] + tel.steer[1:] + 0.5 * np.diff(yaw)

if pred is not None:
    practice("largest miss of the rule (rad)", np.abs(wrap_angle(direction_angle(D) - pred)).max(), 0.0, tol=1e-12)

The wheel angle is the one logged for the *end* of the step. Try `tel.steer[:-1]` in place of `tel.steer[1:]` and the
rule misses by 0.011 rad.

Now split every step arrow into its forward and left parts with your `forward_left`, and look at the two columns. The
car moves 0.4 m a step, nearly all of it forward, and only a little to the side.

**Guess first:** will the left part of a step be closer to 0.4 m, to 0.02 m, or to 0.0001 m?

In [ ]:
try:
    fl = forward_left(yaw[:-1], D)
    practice("mean forward per step (m)", fl[:, 0].mean(), 0.3998, tol=1e-4)
    practice("largest left per step (m)", fl[:, 1].max(), 0.0164, tol=1e-4)
except NotImplementedError:
    print("Write forward_left above first, then run this cell again.")

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/move_vs_point.png" alt="Over the gentle lap, the angle from the car&#x27;s yaw to its GPS step, and the steering angle plus half the step&#x27;s turn, drawn as two curves against the time into the lap. They lie on top of each other. The angle between where the car moves and where it points is at most 0.0091 rad on the straights, about 0.036 rad through the middles of both bends, and 0.041 rad at most anywhere.">
<figcaption><em>Lecture figure 16. Over the gentle lap, the angle from the car&#x27;s yaw to its GPS step, and the steering angle plus half the step&#x27;s turn, drawn as two curves against the time into the lap. They lie on top of each other. The angle between where the car moves and where it points is at most 0.0091 rad on the straights, about 0.036 rad through the middles of both bends, and 0.041 rad at most anywhere.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/move_point_diagram.png" alt="Three directions drawn from the front axle, east at the bottom for reference: the body&#x27;s yaw θ, the front wheel at θ + δ, and the chord of the step at θ + δ + ½Δθ. The angles are drawn far larger than they are, and not to scale with each other: at step 700, δ is 0.0338 rad and ½Δθ is 0.0025 rad.">
<figcaption><em>Lecture figure 17. Three directions drawn from the front axle, east at the bottom for reference: the body&#x27;s yaw θ, the front wheel at θ + δ, and the chord of the step at θ + δ + ½Δθ. The angles are drawn far larger than they are, and not to scale with each other: at step 700, δ is 0.0338 rad and ½Δθ is 0.0025 rad.</em></figcaption>
</figure>

## 7. Break it, fix it

Each cell below holds a teammate's function with one bug. Run it, read what `practice` expected, find the bug and fix it.

**Bug 1.** A teammate's `wrap` gives the wrong answer for $-3.5$ (it should be $+2.783185$). Find out why.

In [ ]:
def teammate_wrap(a):
    return math.fmod(a + math.pi, 2 * math.pi) - math.pi

practice("teammate's wrap of -3.5", teammate_wrap(-3.5), 2.783185, tol=1e-6)

**Bug 2.** A teammate computed the left part for every step, but the largest should be 0.0164 m.

In [ ]:
def teammate_left(theta, v):
    h = np.stack([np.cos(theta), np.sin(theta)], axis=-1)
    return v[..., 0] * h[..., 1] - v[..., 1] * h[..., 0]

practice("largest left per step (m)", teammate_left(yaw[:-1], D).max(), 0.0164, tol=1e-4)

**Bug 3.** A different teammate dotted each step with the left arrow, written as `(sin, cos)`. The largest left part
should again be 0.0164 m.

In [ ]:
def teammate_left_arrow(theta):
    return np.stack([np.sin(theta), np.cos(theta)], axis=-1)

practice("largest left per step (m)", (D * teammate_left_arrow(yaw[:-1])).sum(axis=-1).max(), 0.0164, tol=1e-4)

**Bug 4.** The playground offers a wrong answer to Exercise A under the preset name `naive_arctan`: the lecture's first
try, `np.arctan(dy / dx)`. Run it there and read the verdict: it misses the yaw by more than 0.1 rad on 888 steps,
and its worst miss is 3.142 rad. Then fix it.

The next lesson turns these four functions into a change of frame:

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/next_lesson.png" alt="The car at the start of gentle&#x27;s first bend, seen from above, with the yellow centre line and the white edge line running ahead of it into the bend. A question mark asks where each line is, in metres ahead of the car and to its left.">
<figcaption><em>Lecture figure 18. The car at the start of gentle&#x27;s first bend, seen from above, with the yellow centre line and the white edge line running ahead of it into the bend. A question mark asks where each line is, in metres ahead of the car and to its left.</em></figcaption>
</figure>

## 8. Stretch: the challenge problems

**Challenge 1.** Take the car's path on the `straight` road, add a random error of standard deviation
$\sigma = 0.05$ m to each coordinate of each position, and take the direction from the fix at step $i$ to the fix at step
$i + k$. The lecture shows the spread of the direction's error is about $\sqrt{2}\sigma / (0.4k)$. The cell simulates it
for $k = 18$ over 500 random seeds.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.1-positions-and-headings/figures/gps_law.png" alt="The standard deviation of the yaw error against the fix spacing k, from 1 to 40, on the straight road with sigma 0.05 m. A curve shows the formula, the square root of 2 times sigma over 0.4 k, and dots show the simulation over 500 seeds; the dots sit on the curve. A dashed line marks 0.01 rad, which the curve crosses between k = 17 and k = 18.">
<figcaption><em>Lecture figure 19. The standard deviation of the yaw error against the fix spacing k, from 1 to 40, on the straight road with sigma 0.05 m. A curve shows the formula, the square root of 2 times sigma over 0.4 k, and dots show the simulation over 500 seeds; the dots sit on the curve. A dashed line marks 0.01 rad, which the curve crosses between k = 17 and k = 18.</em></figcaption>
</figure>

In [ ]:
sigma, k = 0.05, 18
straight = run(Car(), "straight")
fixes = np.stack([straight.x, straight.y], axis=-1)
errors = []
for seed in range(500):
    noisy = fixes + np.random.default_rng(seed).normal(0.0, sigma, fixes.shape)
    errors.append(None)  # your code: wrap_angle(direction_angle(noisy[k:] - noisy[:-k]) - straight.yaw[:-k])

if errors[0] is not None:
    practice("spread of the yaw error at k = 18 (rad)", np.concatenate(errors).std(), 0.00980, tol=1e-4)

**Challenge 2.** Write `atan2_by_hand(v)` using only `np.arctan` of the slope and `np.where`. The cell compares it with
`np.arctan2` on four probe arrows, (0, 1), (0, -1), (-1, 0) and (0, 0), then on 30 seeded arrows. The zero arrow has no
direction, so its answer must be `nan`.

In [ ]:
def atan2_by_hand(v):
    """The direction of v from np.arctan and np.where only: all four quarters, +-pi/2 when v_x = 0, nan for (0, 0)."""
    return None  # your code

probes = np.vstack([[[0.0, 1.0], [0.0, -1.0], [-1.0, 0.0], [0.0, 0.0]], np.random.default_rng(1111).normal(0, 5, (30, 2))])

if atan2_by_hand(probes) is not None:
    want = np.where((probes == 0).all(axis=1), np.nan, np.arctan2(probes[:, 1], probes[:, 0]))
    practice("atan2_by_hand matches np.arctan2 on all 34 probes, nan for the zero arrow", float(np.allclose(atan2_by_hand(probes), want, rtol=0, atol=1e-12, equal_nan=True)), 1.0, tol=0)

## What's next

You can now put any position and direction on the road's map, and say how far ahead of the car and how far to its left
an object is. In the next lesson, *The view from the driver's seat*, you use these parts to write every lane line in the
car's own coordinates, so the camera can draw it.